In [0]:


from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import *
# sample data for dataframe
data_list =[("Ravi", 28,1,2002),
            ("Raj", 20,12,2001),
            ("Ravi", 28,1,2002),
            ("Raj", 20,12,2000)]
#-- create df with data and show the schema
raw_df= spark.createDataFrame(data_list)
# -- show df
raw_df.show()
# -- print schema
raw_df.printSchema()

#-- Case 1: regenerate the dataframe with the correct schema
# define the schema for the data
raw_df= spark.createDataFrame(data_list).toDF("Name","Date","Month","Year")
# -- show df
raw_df.show()
# -- print schema
raw_df.printSchema()

# Case 2: regenerate the dataframe with the correct schema
# fix the datatype to string instead of long
data_list =[("Ravi", "28","1","2002"),
            ("Raj", "20","12","2001"),
            ("Ravi", "28","1","2002"),
            ("Raj", "20","12","2000")]

raw_df= spark.createDataFrame(data_list).toDF("Name","Date","Month","Year")
# -- show df
raw_df.show()
# -- print schema
raw_df.printSchema()

# Case-4 creating universal uniq number monotonically_increasing_id and uuid
uniq_df1 = raw_df.withColumn("ID", monotonically_increasing_id())  # start with zero
uniq_df1.show()
uniq_df1 = raw_df.withColumn("ID", monotonically_increasing_id()+100)  # start with number you mentioned
uniq_df1.show()

## UUID
uuid_df1 = raw_df.withColumn("ID",expr("uuid()") ) # express universal id
uuid_df1.show()

# Case where we modify the dataframe with incorrect years work with Case_When_Then scenarios
data_list =[("Ravi", "28","1","2002"),
            ("Raj", "20","12","01"),  # 2001
            ("Ravi", "28","1","99"), # 1999
            ("Aavi", "28","1","9"),  # 2009
            ("Raj", "20","12","80")]  # 1982
raw_df= spark.createDataFrame(data_list).toDF("Name","Date","Month","Year")
raw_df.show()

# modify to correct the year problem 
# use case when else along with cast function. 
mod_dt = raw_df.withColumn("Year", when(raw_df.Year < 100, lit(2000)+raw_df.Year).otherwise(raw_df.Year))
mod_dt.show()
# cast at every field value
mod_dt = raw_df.withColumn("Year", expr("""case when cast(year as int) < 21 then cast (year as int) + 2000
                                        when cast(year as int) < 100 then cast( year as int) + 1900
                                        else cast (year as int)
                                        end"""))
mod_dt.show()

# cast at the end of the field mofications. - Bast way to do. 
mod_dt1 = raw_df.withColumn("Year", expr("""case when year < 21 then year + 2000
                                        when year < 100 then year + 1900
                                        else year
                                        end""").cast("int"))
mod_dt1.show()

mod_dt1.printSchema()  # shows year got converted to integer at schema level

# case correct the original schema alone with correct data types to avoid cast

data_list =[("Ravi", "28","1","2002"),
            ("Raj", "20","12","01"),  # 2001
            ("Ravi", "28","1","99"), # 1999
            ("Aavi", "28","1","9"),  # 2009
            ("Raj", "20","12","80")]  # 1982
raw_df= spark.createDataFrame(data_list).toDF("Name","Date","Month","Year")
raw_df.printSchema()

fix_df = raw_df.withColumn("Year",col("Year").cast("int")) \
                .withColumn("Month",col("Month").cast("int")) \
                .withColumn("Date",col("Date").cast("int"))
fix_df.printSchema()

# cast at the end of the field modification are not needed. - Bast way to do. - no need to cast in this case
mod_dt2 = fix_df.withColumn("Year", when (col("year") < 21, col("Year")+2000)\
                                    .when (col("year") < 100, col("Year")+1900) \
                                    .otherwise(col("Year")))
mod_dt2.show()
mod_dt2.printSchema()

# add a new column DOB using exisiting value
mod_dt2 = mod_dt2.withColumn(
    "DOB",
    expr(
        "to_date(concat(lpad(date,2,'0'),'-',lpad(month,2,'0'),'-',year),'dd-MM-yyyy')"
    )
)
mod_dt2.show()
mod_dt2.printSchema()
